# End-to-End RAG Pipeline: From Documents to Answers

**Track:** LLMOps  
**Toolchain:** sentence-transformers - ChromaDB - tiktoken  
**Objective:** Build a complete Retrieval-Augmented Generation pipeline from scratch, understanding every step: loading, chunking, embedding, indexing, retrieval, and generation.

---

## What is RAG and Why Does Every AI Product Use It?

### The Problem RAG Solves

LLMs have a knowledge cutoff (trained on data up to a certain date). They also don't know YOUR company's private data. Without RAG:
- "What's our Q4 revenue?" -> "I don't have access to your financial data" (useless)
- "What changed in policy v3.2?" -> Hallucinated answer (DANGEROUS)

With RAG:
- Step 1: RETRIEVE relevant documents from your database
- Step 2: AUGMENT the prompt with those documents as context
- Step 3: GENERATE an answer grounded in real data

```
User Question: "What's our Q4 revenue?"
       |
[1. RETRIEVE] --> Search vector DB --> Find: "Q4 2025 revenue: $58.1M"
       |
[2. AUGMENT]  --> Prompt: "Given this context: [Q4 revenue doc], answer: What's our Q4 revenue?"
       |
[3. GENERATE] --> LLM: "Based on the financial report, Q4 2025 revenue was $58.1 million."
```

### Why Not Just Put Everything in the Prompt?

| Approach | Token Limit | Cost per Query | Latency |
|----------|:-:|:-:|:-:|
| Stuff everything in prompt | 128K tokens max | $0.50+ | 10-30s |
| RAG (retrieve top 5 chunks) | ~2K tokens | $0.01 | 2-5s |

RAG is **50x cheaper** and **5x faster** because you only send relevant chunks, not your entire knowledge base.


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Basic 'Naive RAG' (chunk -> embed -> search) looks great in a demo but fails miserably on complex questions. Seniors build Advanced RAG pipelines featuring query rewriting, re-ranking, and metadata filtering.

**Mental Model:** RAG is an open-book exam. You (Retriever) go to the library, find the 5 most relevant textbook pages, and hand them to the student (LLM) to formulate the final answer based ONLY on those pages.

**Common Junior Pitfall:** Chunking documents blindly every 500 words, chopping a crucial paragraph in half. Seniors use semantic chunking, keeping related sentences together and prepending document titles to retain context.

---


In [ ]:
# Cell 1 -- Install
!pip install -q chromadb sentence-transformers tiktoken numpy


---
## Step 1: Document Loading & Chunking

### Why Chunking Matters

Documents are too long to embed as a single vector. You split them into chunks that each contain ONE coherent idea. The chunk size affects everything:

| Chunk Size | Retrieval Precision | Context Quality | Token Cost |
|:-:|:-:|:-:|:-:|
| 100 tokens (tiny) | Very high (exact match) | Low (too little context) | Low |
| 256 tokens | High | Good | Medium |
| 512 tokens (recommended) | Good | Very good | Medium |
| 1024 tokens | Lower | Most context | Higher |
| Full document | Lowest | Everything but noisy | Highest |

### Chunking Strategies

| Strategy | How It Works | Best For |
|----------|-------------|----------|
| Fixed-size | Split every N characters | Simple, fast |
| Sentence-based | Split at sentence boundaries | Better coherence |
| Recursive | Split by paragraph -> sentence -> word | Best quality |
| Semantic | Split when topic changes (using embeddings) | Highest quality, expensive |


In [ ]:
# Cell 2 -- Document chunking
import tiktoken

# Simulate company documents
documents = [
    {'title': 'Q4 Financial Report', 'content': 'Quarter 4 2025 Results Summary. Total revenue reached $58.1 million, representing a 23% year-over-year increase. The AI products division contributed $21.3 million, up 67% from last year. Operating expenses were $42.7 million, resulting in an operating margin of 26.5%. The company ended the quarter with $127 million in cash reserves. Customer acquisition cost decreased to $340 from $450 in Q3, driven by improved organic growth. Net new customers: 1,247 enterprise accounts.'},
    {'title': 'Engineering Policy v3.2', 'content': 'All production deployments require: (1) Automated test coverage above 80%. (2) Security scan with zero critical vulnerabilities. (3) Performance benchmarks showing no regression beyond 5%. (4) Two peer reviews from senior engineers. (5) Rollback plan documented in the deployment ticket. Changes from v3.1: Added requirement for AI model bias testing before deployment. New SLA: 99.95% uptime for Tier-1 services. Incident response time reduced from 30 to 15 minutes for critical issues.'},
    {'title': 'Product Roadmap 2026', 'content': 'H1 2026 focus areas: Launch multi-modal search across documents, images, and audio. Expand AI agent capabilities to handle customer support tickets end-to-end. Integrate with 15 new enterprise tools via MCP protocol. Target: reduce customer time-to-resolution by 40%. H2 2026: Self-hosted LLM deployment option for enterprise customers with data residency requirements. Advanced analytics dashboard with real-time model performance metrics.'},
]

# Tokenizer for counting tokens
enc = tiktoken.get_encoding('cl100k_base')

def chunk_text(text, chunk_size=150, overlap=30):
    """Split text into overlapping chunks by token count.
    Overlap ensures context at chunk boundaries isn't lost."""
    tokens = enc.encode(text)
    chunks = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk_tokens = tokens[i:i + chunk_size]
        chunk_text = enc.decode(chunk_tokens)
        chunks.append(chunk_text)
        if i + chunk_size >= len(tokens):
            break
    return chunks

all_chunks = []
for doc in documents:
    chunks = chunk_text(doc['content'], chunk_size=150, overlap=30)
    for i, chunk in enumerate(chunks):
        all_chunks.append({'text': chunk, 'source': doc['title'], 'chunk_id': f"{doc['title']}_chunk_{i}"})

print(f'Documents: {len(documents)}')
print(f'Total chunks: {len(all_chunks)}')
for c in all_chunks:
    token_count = len(enc.encode(c['text']))
    print(f"  [{c['source']}] {token_count} tokens: \"{c['text'][:60]}...\"")


---
## Step 2: Embedding & Indexing

### How Vector Search Works (In Simple Terms)

1. Each chunk is converted to a **vector** (list of 384 numbers)
2. Vectors that are about similar topics point in similar directions
3. To find relevant chunks, convert the QUESTION to a vector too
4. Find the chunks whose vectors are closest (cosine similarity)

```
"Q4 revenue" -> vector [0.23, -0.14, 0.87, ...]
"Financial results summary" -> vector [0.21, -0.12, 0.85, ...]  <- SIMILAR! (cos_sim = 0.97)
"Engineering policy" -> vector [-0.45, 0.33, 0.12, ...]         <- DIFFERENT (cos_sim = 0.12)
```


In [ ]:
# Cell 3 -- Embed and index with ChromaDB
import chromadb
import numpy as np

# Create ChromaDB collection (vector database)
client = chromadb.Client()
collection = client.create_collection(
    name='company_docs',
    metadata={'hnsw:space': 'cosine'}  # Use cosine similarity for search
)

# Add all chunks to the vector store
collection.add(
    documents=[c['text'] for c in all_chunks],
    metadatas=[{'source': c['source']} for c in all_chunks],
    ids=[c['chunk_id'] for c in all_chunks],
)

print(f'Indexed {collection.count()} chunks in ChromaDB')
print(f'Each chunk is embedded as a {384}-dimensional vector')
print(f'\nWhat just happened:')
print(f'  1. ChromaDB used its built-in embedding model')
print(f'  2. Each chunk -> 384-dim vector')
print(f'  3. Vectors indexed in HNSW graph for fast nearest-neighbor search')
print(f'  4. Ready for semantic queries!')


---
## Step 3: Retrieval

### Retrieval Strategies

| Strategy | How It Works | Quality | Speed |
|----------|-------------|:-:|:-:|
| Dense retrieval | Embedding similarity (what we're doing) | High | Fast |
| Sparse retrieval | BM25 keyword matching | Medium | Very fast |
| Hybrid | Dense + Sparse + reranking | Highest | Slower |

### How Many Chunks to Retrieve? (top_k)

| top_k | Context Quality | Risk | Cost |
|:-:|---|---|:-:|
| 1 | May miss relevant info | High | Lowest |
| 3 | Good balance | Medium | Low |
| 5 (recommended) | Comprehensive | Low | Medium |
| 10 | Very comprehensive | Very low (but adds noise) | Higher |


In [ ]:
# Cell 4 -- Query the vector store

test_queries = [
    'What was the Q4 revenue?',
    'What are the deployment requirements?',
    'What is planned for H2 2026?',
]

for query in test_queries:
    results = collection.query(query_texts=[query], n_results=2)
    print(f'\nQuery: "{query}"')
    for i, (doc, meta, dist) in enumerate(zip(results['documents'][0], results['metadatas'][0], results['distances'][0])):
        similarity = 1 - dist  # ChromaDB returns distance, we want similarity
        print(f'  Result {i+1} (sim={similarity:.3f}, source={meta["source"]}):') 
        print(f'    "{doc[:100]}..."')


---
## Step 4: Augmented Generation

### The RAG Prompt Template

This is where retrieval connects to generation. The template MUST:
1. Clearly separate CONTEXT from QUESTION
2. Instruct the model to ONLY use provided context
3. Tell the model to say "I don't know" if context is insufficient

### Production Reality: Why "I Don't Know" Is Better Than Hallucination

In production, a confident wrong answer is FAR more dangerous than no answer:
- Wrong financial figure -> Bad business decision -> Revenue loss
- Wrong policy answer -> Compliance violation -> Legal penalty
- "I don't have that information" -> User asks a human -> Correct answer


In [ ]:
# Cell 5 -- Complete RAG pipeline

def rag_pipeline(question, collection, top_k=3):
    """Complete RAG: Retrieve -> Augment -> Generate (simulated)."""
    # Step 1: Retrieve
    results = collection.query(query_texts=[question], n_results=top_k)
    context_chunks = results['documents'][0]
    sources = [m['source'] for m in results['metadatas'][0]]

    # Step 2: Augment (build the prompt)
    context = '\n---\n'.join(context_chunks)
    rag_prompt = f"""Answer the question using ONLY the provided context.
If the context does not contain enough information, say: "I don't have sufficient information to answer this."
Do NOT make up facts.

CONTEXT:
{context}

QUESTION: {question}

ANSWER (cite the source document):"""

    # Step 3: Generate (simulated -- in production, send to LLM API)
    # response = litellm.completion(model='gpt-4o', messages=[{'role':'user','content':rag_prompt}])

    token_count = len(enc.encode(rag_prompt))
    return {
        'question': question,
        'context_chunks': len(context_chunks),
        'sources': list(set(sources)),
        'prompt_tokens': token_count,
        'estimated_cost': f'${token_count * 0.000003:.6f}',
        'prompt_preview': rag_prompt[:200] + '...',
    }

# Test the full pipeline
test_questions = [
    'What was Q4 2025 revenue and how much did AI products contribute?',
    'What changed between engineering policy v3.1 and v3.2?',
    'What is the company planning for H2 2026?',
    'Who is the CEO?',  # NOT in our documents -- should trigger "I don't know"
]

for q in test_questions:
    result = rag_pipeline(q, collection)
    print(f'\nQ: "{q}"')
    print(f'  Chunks retrieved: {result["context_chunks"]}')
    print(f'  Sources: {result["sources"]}')
    print(f'  Prompt tokens: {result["prompt_tokens"]} (cost: {result["estimated_cost"]})')


---
## RAG Quality Issues & How to Fix Them

| Problem | Symptom | Fix |
|---------|---------|-----|
| **Low recall** | Relevant docs not retrieved | Increase top_k, use hybrid search |
| **Low precision** | Irrelevant docs retrieved | Better chunking, add metadata filters |
| **Hallucination** | Answer not in context | Strengthen "only use context" instruction |
| **Stale data** | Outdated information | Add real-time streaming embeddings (NB05) |
| **Wrong chunks** | Right doc, wrong section | Smaller chunks, add overlap |

### Evaluating RAG Quality

Use RAGAS metrics (covered in `18_llm_evaluation.ipynb`):
- **Faithfulness**: Does the answer stick to the context? (no hallucination)
- **Answer Relevancy**: Does it actually answer the question?
- **Context Precision**: Were the retrieved chunks relevant?
- **Context Recall**: Did we find ALL the relevant chunks?

### Production Reality Check

**What we built:** A simple in-memory RAG with ChromaDB and simulated LLM.  
**What production looks like:**
- Document loading from S3, SharePoint, Confluence, Notion
- Chunking with metadata extraction (author, date, section headers)
- Embedding with hosted models (cost: $0.0001 per 1K tokens)
- Vector DB: PgVector (small), Pinecone/Qdrant (large)
- LLM: API call with streaming response (SSE, NB10)
- Monitoring: Track retrieval quality, latency, costs (Langfuse, NB18)


---
## Summary

| RAG Step | What Happens | Key Decisions |
|----------|-------------|---------------|
| 1. Load | Read documents from sources | Format, encoding, metadata |
| 2. Chunk | Split into digestible pieces | Chunk size (512 rec.), overlap (10-20%) |
| 3. Embed | Convert text to vectors | Model choice, dimensions (384/768/1536) |
| 4. Index | Store vectors for fast search | ChromaDB/pgvector/Pinecone |
| 5. Retrieve | Find relevant chunks | top_k (3-5), similarity threshold |
| 6. Generate | LLM answers using context | Grounding prompt, temperature=0 |

**Next -->** Notebooks on LLMOps, Agentic Orchestration, and LLM Evaluation build on this RAG foundation.